# DNN Training Results Analysis

This notebook presents the comprehensive analysis of the results obtained from training various deep neural network configurations. The main focus is evaluating the impact of hyperparameters (Learning Rate, Activation Function, Dropout, and Optimizer) on the final model performance, measured primarily through `Validation Accuracy`.

## 1. Performance Distribution by Hyperparameter
![Boxplots](plots/1_hyperparams_boxplots.png)

### Analysis of Systematic Behaviors and Outliers:
The boxplot chart offers an immediate and comprehensive view of the marginal effects of each hyperparameter:

- **Activation Function (Systematic Degradation Behavior):** The clearest result of the entire study is evident looking at the top-right panel. It denotes **a systematic behavior where the `sigmoid` function provides extremely poor and unreliable results** compared to modern counterparts. This function typically suffers from the *vanishing gradient* problem, which prevents the network from learning effectively, leading to very low medians and a high variance of weak results (many models remain confined to an accuracy just above random chance). Conversely, **`relu` and `elu` prove to be excellent and systematically winning choices**, exhibiting not only high medians but also a dispersion cloud concentrated in higher values.
- **Learning Rate:** The distributions as a function of LR show the typical "skewed bell" shape. Intermediate values ($10^{-2}$) host almost all the top-performing models. Values that are too low ($10^{-5}$, $10^{-4}$) often struggle to converge (lower median and efficiency with many downward outliers), while values like $10^{-3}$, $10^{-1}$ present numerous **outlier behaviors** in the form of steep drops in performance, caused by unstable gradients or divergent logic.
- **Dropout:** Dropout impacts the global distributions less drastically in this restricted search space. Anyway, slightly better results are seen for dropouts lower than 0.2.
- **Optimizer:** Among the optimizers, slight differences in median value are seen, but all contain very low and very high outliers, often corresponding to malicious configurations (e.g., combined with sigmoid or wrong LRs).

## 2. Learning Rate Sensitivity with respect to the Optimizer
![Val Acc per Optimizer](plots/2_val_acc_by_lr_optimizer.png)

### Motivation of Choices (Optimizer):
This interaction plot highlights the dynamic relationship between optimizer and LR:
- Methods like `Adam` and `RMSprop` tend to have higher accuracy for LRs in the range [$10^{-3}$; $10^{-2}$), while `nesterov` performs well even for higher LRs ($[10^{-2}; 0.5]$).
- **Outliers and Numerical Instability:** For high LRs, methods like `Adam` and `RMSprop` plummet vertiginously, whereas `nesterov` performs better, showing on average faster and computationally smoother convergence. `nesterov` shows an extremely higher variability as both a merit and a flaw: if on the one hand it is much more dependent on other parameters (high variability), on the other it shows many more outliers that allow obtaining high accuracy even with LRs for which other models averagely seem to excel.
Either way, **the optimal choice averagely suggests adopting an LR around $10^{-2}$**.

## 3. Learning Rate Sensitivity with respect to Activation
![Val Acc per Activation](plots/3_val_acc_by_lr_activation.png)

### Reconfirmation of Extreme Systematic Behaviors:
The graph re-proposes what was observed previously, conferring it parametric robustness:
- The trace corresponding to the **`sigmoid` is almost constantly the worst** at every investigated learning rate value. It barely reaches or touches the marginal performance of the other activations in isolated spots where `relu` and `elu` are already chronically collapsing due to instability at high regimes (LR = $10^{-1}$).
- The scalar convergence profiles of **`relu` and `elu` describe a clear parabolic shape in log-scale**, dictating as an empirical gold standard their rapid implementation for the vast majority of feed-forward network tasks. Maintaining the `relu`/`elu` activation is a fundamental prerequisite in this project to raise the maximum accuracy benchmark.

## 4. Top 5 Configurations Analysis
![Top 5 Models](plots/4_top_5_models.png)

### Motivation of the Excellent Winners:
The close comparison among the absolute top 5 in Validation Accuracy with the hold-out metric (Test Accuracy):
- **Winning choices:** As expected, no best-in-class run uses sigmoid or LR outside of $[10^{-3};10^{-1}]$. All align with absolute dominance on `elu` or `relu` activations paired with perfectly centered LRs.
- **Robust Generalization (Absence of Overfitting):** The elite configurations record an insignificant gap, in some neighbors almost invisible, between Validation and Test Accuracy. On average, as seen, the applied topology optimally regularizes the network; nevertheless, exactly 3 models among the top 5 have `nesterov` as the optimizer: these results configure themselves, as seen, as *statistical outliers*.

## 5. Local Sensitivity Analysis (The Top 5 Models)
In the subsections below, we analyze how accuracies vary by shifting, for each of the top 5 models (keeping all other factors constant), one unknown parameter at a time (starting and comparing from the optimal setup indicated by a red star on the graph).

### Top 1 Model Analysis
![Top 1 Analysis](plots/5_top1_model_analysis.png)

### Top 2 Model Analysis
![Top 2 Analysis](plots/5_top2_model_analysis.png)

### Top 3 Model Analysis
![Top 3 Analysis](plots/5_top3_model_analysis.png)

### Top 4 Model Analysis
![Top 4 Analysis](plots/5_top4_model_analysis.png)

### Top 5 Model Analysis
![Top 5 Analysis](plots/5_top5_model_analysis.png)

### Neighborhood Explorations (Robustness and Drops):
- **Activation function stability:** The `sigmoid` is confirmed again as the worst *activation function*. The sigmoid is the latent variable of undeniable deterioration of the observed performances.
- **Learning rate sensitivity:** In many of such slices (section of the variation gradient at a single degree of freedom), the impetuous variances of the chosen `optimizer` or `learning_rate` generate precipitous drops in validation accuracy.
- The highlighted key lesson reiterates that the empirical success of a deep network lies in a narrow *sweet-spot* or unified domain: foolishly perturbing a calibrated synergy of a Top Model (perhaps changing the optimizer in favor of the insidious and primitive SGD on that range) will cause a drastic decrease. The best exposed choices demonstrate the inviolability of the calibrated trade-off between a non-blocking function like **relu/elu** combined with an optimizer like **Adam/RMSprop** with an LR in the range [$10^{-3}$; $10^{-2}$), while (activation function being equal) `nesterov` performs well even for higher LRs ($[10^{-2}; 0.5]$) but the latter in a very unstable way giving good results only as an outlier.